# Tarn — train the diagram detector on a Colab GPU (P14)

Trains the shape-and-arrow detector (RT-DETR R18, Apache-2.0) on the bundle made by
`tarn diagram data pack` (FC database, Flowchart 3b and Tarn's synthetic drawings: public and
synthetic material only, no student data).

1. Upload `var/diagrams/colab-bundle.zip` to Google Drive, into `MyDrive/tarn/`.
2. Runtime → Change runtime type → **T4 GPU**.
3. Run the cells in order. The model folder is written to Drive after every epoch, so if the
   session disconnects, reconnect and run the cells again: training resumes.
4. Download `shape-detector-v1.zip` (last cell) and unzip it into
   `var/models/diagram/shape-detector-v1/` on the development machine, then run
   `tarn diagram bench`.


In [ ]:
!nvidia-smi
import sys
assert sys.version_info >= (3, 12), "Tarn's code needs Python 3.12 or newer"


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

BUNDLE = "/content/drive/MyDrive/tarn/colab-bundle.zip"
OUT = "/content/drive/MyDrive/tarn/shape-detector-v1"   # persistent: checkpoints and the model
EPOCHS = 40        # passes over the mixed sources
STEPS = 500        # optimiser steps per epoch
BATCH = 8          # images per step (fits a T4 in fp16 at 1024 px)


In [ ]:
# Unpack to the local disk (much faster to read than Drive)
!mkdir -p /content/tarn && unzip -q -o "$BUNDLE" -d /content/tarn
!ls /content/tarn /content/tarn/code && wc -l /content/tarn/data/manifest.jsonl


In [ ]:
# Colab already has torch, OpenCV and NumPy; pin transformers to the version Tarn uses
!pip install -q "transformers==5.18.0"


In [ ]:
!cd /content/tarn && PYTHONPATH=/content/tarn/code python -m tarn_adapters.diagram.colab \
    --data /content/tarn/data --out "$OUT" \
    --epochs $EPOCHS --steps $STEPS --batch $BATCH --workers 2


In [ ]:
import json, pathlib
manifest = json.loads(pathlib.Path(OUT, "tarn-detector.json").read_text())
for h in manifest["history"]:
    print(h)


In [ ]:
# The published model only (no optimiser checkpoint): download this zip
import shutil, tempfile, pathlib
keep = [p for p in pathlib.Path(OUT).iterdir() if p.is_file() and p.name != "checkpoint.pt"]
tmp = pathlib.Path(tempfile.mkdtemp()) / "shape-detector-v1"
tmp.mkdir()
for p in keep:
    shutil.copy(p, tmp / p.name)
archive = shutil.make_archive("/content/shape-detector-v1", "zip", tmp)
print(archive, [p.name for p in keep])
from google.colab import files
files.download(archive)
